In [1]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import train_test_split

import torchvision
from torchvision.datasets import MNIST
import torchvision.transforms as transformer

from sklearn.metrics import confusion_matrix


In [2]:
transforms = transformer.Compose([
    transformer.ToTensor(),
    transformer.Normalize((0.5,),(0.5,))
])

In [3]:
train_dataset = MNIST(root = "./mndata", train = True, download = True, transform = transforms )
test_dataset = MNIST(root = "./mndata", train = False, download = True, transform = transforms)

In [4]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32)

In [5]:
#CNN MODEL
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1, 28, kernel_size = 2, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(28, 56, kernel_size = 2, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

           # nn.Conv2d(56, 112, kernel_size = 2, padding = 1),
            #nn.ReLU(),
            #nn.MaxPool2d(2,2),

        )
        self.fc_layer = nn.Sequential(
            nn.Linear(7*7*56,10),
            nn.ReLU(),
        )

    def forward(self,x):
        x = self.model(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layer(x)

        return x

In [6]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [25]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for img, label in train_loader:
        optimizer.zero_grad()

        img = img.squeeze()
        outputs = model(img)
        
        loss  = criterion(outputs, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs} training loss = {epoch_loss}")

epoch = 1/20 training loss = 0.16282833073536554
epoch = 2/20 training loss = 0.16650884200781585
epoch = 3/20 training loss = 0.1600260091068844
epoch = 4/20 training loss = 0.15609039066235225
epoch = 5/20 training loss = 0.15925358116353552
epoch = 6/20 training loss = 0.15192946920419734
epoch = 7/20 training loss = 0.16813344180012743
epoch = 8/20 training loss = 0.14958325658738614
epoch = 9/20 training loss = 0.16566000586102406
epoch = 10/20 training loss = 0.16910696588009597
epoch = 11/20 training loss = 0.1402828646145761
epoch = 12/20 training loss = 0.14657771482666335
epoch = 13/20 training loss = 0.15339407741824787
epoch = 14/20 training loss = 0.14702053302327792
epoch = 15/20 training loss = 0.14652004525214435
epoch = 16/20 training loss = 0.15411126475359002
epoch = 17/20 training loss = 0.15504105864490073
epoch = 18/20 training loss = 0.1355478931973378
epoch = 19/20 training loss = 0.15119337276269992
epoch = 20/20 training loss = 0.15245694223145645


In [26]:
model.eval()
predictions = []
labels = []

with torch.no_grad():
    total = 0
    correct = 0

    for img, label in test_loader:
        img = img.squeeze()
        outputs  = model.forward(img)
       
        _,predicted = torch.max(outputs, 1)

        correct += (predicted==label).sum().item()
        total += label.size(0)

        predictions.extend(predicted.numpy())
        labels.extend(label.numpy())
    
    print("Accuracy : ", correct/total*100,"%")
    print("Confusion Matrix :\n",confusion_matrix(labels,predictions))

Accuracy :  95.6 %
Confusion Matrix :
 [[ 961    0    1    0    1    2    6    1    8    0]
 [   0 1110    2    3    5    1    2    4    8    0]
 [   2    2 1004    4    0    0    4    9    7    0]
 [   0    2   10  955    0   12    0   12    4   15]
 [   2    1    3    0  905    1   13    0    7   50]
 [   3    8    4   14    7  837    6    4    3    6]
 [   3    4    5    0    4   19  918    0    5    0]
 [   0    2   14    0    2    0    1 1001    4    4]
 [   3    2    8    2    3    6   10    4  926   10]
 [   1    6    4    6    9    8    1   27    4  943]]


In [27]:
import pandas as pd
predicted_df = pd.DataFrame(predicted.numpy(), columns = ["Predicted vals"])
actual_df = pd.DataFrame(label.numpy(), columns = ["Actual vals"])

pd.concat([predicted_df,actual_df], axis = 1)

,Predicted vals,Actual vals
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5
5,6,6
6,7,7
7,8,8
8,9,9
9,0,0


RNN model

In [28]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 64, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out,_ = self.rnn(x,h0)
        out = self.fc(out[:,-1,:])
        return out

In [29]:
MNIST

torchvision.datasets.mnist.MNIST

In [30]:
input_size = 28
model = RNN(input_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [31]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb = xb.squeeze(1)
        outputs = model(xb)
        #outputs = outputs.squeeze()
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"epoch {epoch+1}/{epochs} loss = {running_loss}")

epoch 1/10 loss = 1593.6509481966496
epoch 2/10 loss = 808.7577631808817
epoch 3/10 loss = 634.6650590002537
epoch 4/10 loss = 540.717595949769
epoch 5/10 loss = 509.5687805879861
epoch 6/10 loss = 474.9795943722129
epoch 7/10 loss = 455.9517270512879
epoch 8/10 loss = 439.43794373236597
epoch 9/10 loss = 411.9140745373443
epoch 10/10 loss = 417.5290956450626


In [33]:
model.eval()
predictions = []
labels = []
with torch.no_grad():
    total = 0
    correct = 0

    for xb, yb in test_loader:
        xb = xb.squeeze(1)
        outputs = model(xb)

        _,predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

        predictions.extend(predicted.numpy())
        labels.extend(yb.numpy())
    print("Accuracy : ", correct/total*100,"%")
    print("Confusion Matrix :\n", confusion_matrix(labels,predictions))

Accuracy :  94.12 %
Confusion Matrix :
 [[ 945    0    3    1    7    0   16    1    5    2]
 [   0 1098    1    7    1    3    3    1   19    2]
 [   4    1  995    5    6    1    5   10    3    2]
 [   1    1   17  955    0   20    0   11    3    2]
 [   1    2    3    0  911    5   20    1    2   37]
 [  10    5    2   44    4  765   17    1   21   23]
 [  11    1    2    0    7    5  928    0    4    0]
 [   2    5   12    9    1    2    0  981    3   13]
 [   6    1   23    2    2   11   13    2  908    6]
 [   4    6    2    7   22   29    1    9    3  926]]
